# Step 1: 环境准备与数据获取

---

## 目标

把 Python 运行环境搭好，拿到第一份股票行情数据，学会怎么把数据读进来看一眼。

做完这一步你应该能回答三个问题:
1. 我的环境能不能跑 pandas？
2. 数据里有多少只股票、多少天？
3. 数据里有 ST 或科创板股票吗？成交量是手还是股？

## 输入

- Python 3.9+
- 项目源码已拉到本地
- workflow/sample_data.csv (3只股票 x 3天)


In [ ]:
import sys
print(f"Python 版本: {sys.version.split()[0]}")

import pandas as pd
import numpy as np
print(f"pandas: {pd.__version__}, numpy: {np.__version__}")

import sys, os
sys.path.append(os.path.abspath(".."))
from workflow.helpers import (
    is_limit_up, is_limit_down, print_step,
    check_data_quality, estimate_amount
)
print("import 全部成功")


In [ ]:
data_path = "sample_data.csv"
df = pd.read_csv(data_path, encoding="utf-8")
print(f"数据维度: {df.shape[0]} 行 x {df.shape[1]} 列")


In [ ]:
df.head()


In [ ]:
df.info()


In [ ]:
print("股票列表:")
print(df[["代码", "股票名"]].drop_duplicates().to_string(index=False))


In [ ]:
qc = check_data_quality(df, code_col="代码")
for k, v in qc.items():
    if isinstance(v, dict):
        print(f"{k}:")
        for kk, vv in v.items():
            print(f"  {kk}: {vv}")
    else:
        print(f"{k}: {v}")


In [ ]:
df["成交额_亿"] = df.apply(
    lambda r: round(estimate_amount(r["成交量"], (r["开盘"] + r["收盘"]) / 2), 2),
    axis=1
)
df[["日期", "代码", "股票名", "成交量", "成交额_亿"]]


## 输出

这一步拿到了一个 pd.DataFrame，3只股票 x 3天 = 9行数据。

Step 2 会以这个 DataFrame 为输入开始清洗。

## 常见坑

| 坑 | 表现 | 解决办法 |
|---|---|---|
| CSV 编码不对 | 中文乱码 | 加 encoding 参数 |
| 成交量单位搞混 | 是手还是股 | 看数据源文档 |
| 代码有引号前缀 | 字符串夹带符号 | .str.strip() |
| 日期列不是时间类型 | 无法排序 | pd.to_datetime() |


## 我的理解

这段就三件事: 装包 -> 读数据 -> 看长什么样。

大部分策略错误都出在"数据还没看清就开始写策略"——没发现里面有 ST，或者成交量理解错了。核心习惯: **拿到数据先看行数、列数、唯一股票数**，确认和预期一致再往后走。
